# colab_00 — Setup + environment capture

First notebook of the `ad-glia-fm-prep` execution phase. Mounts Drive, clones the repo, installs `requirements_integration.txt`, captures the resolved software stack into `outputs/software_versions/`, and runs preemptive Audits F (storage budget) and B (APOE metadata presence) before any count-matrix download.

**Order of operations:**

1. Mount Drive (section 1)
2. Clone/pull repo + install env (section 2)
3. Env capture — pip freeze + Python/CUDA/GPU/OS (section 3)
4. Audit F — storage budget table (section 4)
5. Audit B preemptive — donor metadata only, confirm APOE column (section 5)
6. Handoff to `colab_01_li2025_qc.ipynb` (section 6)

No bulk data download in this notebook beyond per-study donor-metadata files (typically <10 MB).

## 1 — Mount Google Drive

Drive persists across runtime resets; later notebooks write large intermediates (h5ad, FM caches, embeddings) here. `colab_00` doesn't produce large artifacts itself, but mounting + confirming the project root exists belongs to setup.

### 1a — Mount Drive and confirm project root

Mounts `/content/drive` and ensures `MyDrive/ad-glia-fm-prep/` exists. This is the Drive-side mirror for large data — runtime-local repo at `/content/ad-glia-fm-prep` is separate (committable artifacts live there).

In [ ]:
import os
from google.colab import drive

drive.mount("/content/drive")

DRIVE_ROOT = "/content/drive/MyDrive/ad-glia-fm-prep"
os.makedirs(DRIVE_ROOT, exist_ok=True)
print(f"DRIVE_ROOT = {DRIVE_ROOT}")

## 2 — Clone repo and install integration env

Repo is cloned to runtime-local `/content/ad-glia-fm-prep` (git ops are slow over the Drive FUSE mount). Committable artifacts (`outputs/audit_report.json`, `outputs/software_versions/*`) are written under the repo path so a final `git add + commit + push` from within this session lands them on GitHub.

### 2a — Clone or pull repo

Clones on first run, pulls otherwise. Adds the repo to `sys.path` so `from src.* import ...` works.

In [ ]:
import os, subprocess, sys

REPO_URL = "https://github.com/pavlemic/ad-glia-fm-prep.git"
REPO_PATH = "/content/ad-glia-fm-prep"

if not os.path.exists(REPO_PATH):
    subprocess.run(["git", "clone", REPO_URL, REPO_PATH], check=True)
else:
    subprocess.run(["git", "-C", REPO_PATH, "pull"], check=True)

if REPO_PATH not in sys.path:
    sys.path.insert(0, REPO_PATH)

subprocess.run(["git", "-C", REPO_PATH, "rev-parse", "--short", "HEAD"], check=True)

### 2b — Install requirements_integration.txt

Installs the integration env (scvi-tools 1.4.3, scanpy 1.10.3, scrublet, scib-metrics, harmonypy). Asserts the runtime is Python 3.12 — required by scvi-tools 1.4.x. `torch` / `jax` are TBD-stubs in the pin file; Colab's CUDA-matched wheel resolves them, captured in section 3.

In [ ]:
import sys

assert sys.version_info[:2] == (3, 12), (
    f"Expected Python 3.12, got {sys.version_info[:2]}. "
    "Switch the Colab runtime to one that ships 3.12 before continuing."
)

# Pin numpy first so pip picks numpy-1.x-compatible wheels for pandas, scipy, etc.
# Without this, pip resolves against Colab's pre-installed numpy 2.x, then
# downgrades numpy — leaving a binary ABI mismatch that crashes on import.
!pip install numpy==1.26.4
!pip install -r {REPO_PATH}/requirements_integration.txt

## 3 — Environment capture

Records the *resolved* software stack — pip freeze + Python/CUDA/GPU/OS — to `outputs/software_versions/`. Per [[software-versions-capture]]: the pin file declares what to install (forward-looking); the freeze + env snapshot records what was actually used (backward-looking, paper-ready). Both are committed.

### 3a — pip freeze snapshot

Writes the actually-resolved package set (including transitive deps) to `outputs/software_versions/colab_00_<date>_pip_freeze.txt`.

In [ ]:
import os
from datetime import date

NOTEBOOK_ID = "colab_00"
TODAY = date.today().isoformat()

VERSIONS_DIR = os.path.join(REPO_PATH, "outputs", "software_versions")
os.makedirs(VERSIONS_DIR, exist_ok=True)

FREEZE_PATH = os.path.join(VERSIONS_DIR, f"{NOTEBOOK_ID}_{TODAY}_pip_freeze.txt")
!pip freeze > {FREEZE_PATH}
print(f"Wrote {FREEZE_PATH}")

### 3b — Python / CUDA / GPU / OS snapshot

Captures what `pip freeze` misses: interpreter version, torch's reported CUDA version, GPU model from `nvidia-smi -L`, OS release, NVIDIA driver. Structured `.json` next to the freeze file. This is what a paper's Methods section is reconstructed from.

In [ ]:
import json, platform, subprocess, sys

env_snapshot = {
    "notebook_id": NOTEBOOK_ID,
    "date": TODAY,
    "python_version": sys.version,
    "platform": platform.platform(),
    "os_release": platform.release(),
}

try:
    import torch
    env_snapshot["torch_version"] = torch.__version__
    env_snapshot["torch_cuda_version"] = torch.version.cuda
    env_snapshot["cudnn_version"] = torch.backends.cudnn.version()
    env_snapshot["cuda_available"] = torch.cuda.is_available()
except ImportError:
    env_snapshot["torch_version"] = None

def _run(cmd):
    try:
        return subprocess.run(cmd, capture_output=True, text=True, check=True).stdout.strip()
    except (FileNotFoundError, subprocess.CalledProcessError):
        return None

env_snapshot["gpu"] = _run(["nvidia-smi", "-L"])
env_snapshot["nvidia_driver"] = _run([
    "nvidia-smi", "--query-gpu=driver_version", "--format=csv,noheader"
])
env_snapshot["git_commit"] = _run(["git", "-C", REPO_PATH, "rev-parse", "HEAD"])

ENV_JSON_PATH = os.path.join(VERSIONS_DIR, f"{NOTEBOOK_ID}_{TODAY}_env.json")
with open(ENV_JSON_PATH, "w") as f:
    json.dump(env_snapshot, f, indent=2)

print(json.dumps(env_snapshot, indent=2))
print(f"\nWrote {ENV_JSON_PATH}")

## 4 — Audit F: storage budget

Per `docs/setup_audits.md`, populate the storage budget table **before** any raw download. Drive has ca. 80–90 GB usable; target estimated total <70 GB to leave 10 GB+ headroom. Estimates here are placeholders — fill in from each data source's documented file sizes (Synapse / Zenodo / GEO listing pages).

### 4a — Build storage budget and write to audit_report.json

Replace `None` values with verified estimates per source before running `colab_01`. The audit reports `incomplete` while any row is `None`.

In [ ]:
import json, os

AUDIT_REPORT_PATH = os.path.join(REPO_PATH, "outputs", "audit_report.json")
os.makedirs(os.path.dirname(AUDIT_REPORT_PATH), exist_ok=True)

audit_f = {
    "audit": "F — storage budget",
    "budget_gb": 70,
    "items": [
        {"item": "SEA-AD raw (Allen portal, MTG + DLPFC h5ads)", "est_gb": 13.0, "keep": False, "note": "MTG QC'd h5ad ~5.9 GB confirmed; DLPFC ~6-7 GB est. — delete after per-study processed AnnData saved"},
        {"item": "Haney 2024 raw (GEO GSE254205)",               "est_gb":  0.7, "keep": False, "note": "GSE254205_ad_raw.h5ad.gz — delete after per-study processed AnnData saved"},
        {"item": "Li 2025 raw (GEO GSE237718)",                  "est_gb":  4.7, "keep": False, "note": "GSE237718_RAW.tar — delete after per-study processed AnnData saved"},
        {"item": "Per-study processed AnnData (×3, sparse CSR)", "est_gb":  3.5, "keep": True,  "note": "est. ~2 GB SEA-AD + ~1 GB Li 2025 + ~0.5 GB Haney, post-QC sparse"},
        {"item": "Integrated AnnData (sparse CSR)",              "est_gb":  3.0, "keep": True,  "note": "keep"},
        {"item": "Astro+microglia subset (sparse CSR)",          "est_gb":  0.5, "keep": True,  "note": "keep"},
        {"item": "FM weights (Geneformer + scGPT cached)",       "est_gb":  2.0, "keep": True,  "note": "cached on Colab runtime; Drive footprint minimal if runtime cache persists"},
        {"item": "LoRA adapters (~30MB × 24 at N=3)",            "est_gb":  1.0, "keep": True,  "note": "adapters-only, not full weights"},
        {"item": "Test-set embeddings (~50–100MB × ~30)",        "est_gb":  3.0, "keep": True,  "note": "test-set only by default"},
        {"item": "Logs + checkpoints",                           "est_gb":  0.5, "keep": False, "note": "rotate"},
    ],
}

missing = [r["item"] for r in audit_f["items"] if r["est_gb"] is None]
if missing:
    audit_f["status"] = "incomplete"
    audit_f["note"] = f"{len(missing)} rows TBD — fill before colab_01 download."
    audit_f["missing_rows"] = missing
else:
    total = sum(r["est_gb"] for r in audit_f["items"])
    audit_f["estimated_total_gb"] = total
    audit_f["status"] = "pass" if total < audit_f["budget_gb"] else "fail"

if os.path.exists(AUDIT_REPORT_PATH):
    with open(AUDIT_REPORT_PATH) as f:
        report = json.load(f)
else:
    report = {}
report["audit_f_storage"] = audit_f
with open(AUDIT_REPORT_PATH, "w") as f:
    json.dump(report, f, indent=2)

print(json.dumps(audit_f, indent=2))

## 5 — Audit B preemptive: donor-level APOE metadata

Confirms APOE genotype column presence in each study's **donor metadata** (sample-level, typically <10 MB) before committing to the bulk count-matrix download. Expected per the data source lock:

- SEA-AD: 84 donors, APOE present
- Haney 2024: 15 donors, APOE3/3 and APOE4/4 by design (GEO GSE254205)
- Li 2025: 56 donors, purpose-built E2/E3-3/E4 carrier matching (GEO GSE237718)

Don't guess URLs — paste the verified metadata-file path from each source's listing page into the constants in cell 5a. Cell 5b loads, checks for an APOE column, and writes pass/fail to `audit_report.json`.

### 5a — Set per-study donor-metadata paths

Each entry should be a URL or Drive path to a small sample/donor manifest file (CSV/TSV). Leave as `None` to skip that study and have Audit B report `skipped` for it.

In [ ]:
# Li 2025 (GSE237718): GEO suppl = GSE237718_RAW.tar only — no standalone donor manifest.
# Haney 2024 (GSE254205): GEO suppl = GSE254205_ad_raw.h5ad.gz only — no standalone manifest.
# APOE metadata for both is embedded in the h5ad obs; checked in per-study QC notebooks.
#
# SEA-AD: Allen portal / AWS open access. Donor-level clinical metadata available;
# paste the URL below once confirmed (small CSV, not the full h5ad).
STUDY_METADATA = {
    "SEA-AD":      {"path": None, "source": "Allen portal / AWS open access",  "expected_donors": 84},
    "Haney 2024":  {"path": None, "source": "GEO GSE254205 — no standalone manifest; APOE in h5ad obs", "expected_donors": 15},
    "Li 2025":     {"path": None, "source": "GEO GSE237718 — no standalone manifest; APOE in h5ad obs", "expected_donors": 56},
}

for study, meta in STUDY_METADATA.items():
    status = "set" if meta["path"] else "TBD"
    print(f"{study:12s}  {status:4s}  source: {meta['source']}")


### 5b — Load each manifest, check APOE column, write Audit B

Detects any column whose name contains `apoe` (case-insensitive). Reports n_donors and APOE value distribution per study. Per-study status:

- `pass` — APOE column found, n_donors ≥ expected
- `warn` — APOE column found, n_donors < expected
- `fail` — no APOE column
- `skipped` — path is None

Overall Audit B `pass` requires every non-skipped study to pass.

In [ ]:
import json, os
import pandas as pd

def check_apoe(study, meta):
    if meta["path"] is None:
        return {"study": study, "source": meta["source"], "status": "skipped", "note": "path is None"}
    df = pd.read_csv(meta["path"], sep=None, engine="python")
    apoe_col = next((c for c in df.columns if "apoe" in c.lower()), None)
    donor_col = next(
        (c for c in df.columns if c.lower() in {"donor_id", "subject_id", "individualid", "individual_id", "patient_id"}),
        None,
    )
    if donor_col is None:
        return {"study": study, "source": meta["source"], "status": "warn",
                "note": "no donor_id column found — len(df) would be misleading; inspect columns manually"}
    n_donors = df[donor_col].nunique()
    if apoe_col is None:
        status = "fail"
    elif n_donors < meta["expected_donors"]:
        status = "warn"
    else:
        status = "pass"
    return {
        "study": study,
        "source": meta["source"],
        "path": meta["path"],
        "n_donors": int(n_donors),
        "expected_donors": meta["expected_donors"],
        "apoe_column": apoe_col,
        "apoe_value_counts": (df[apoe_col].astype(str).value_counts().to_dict() if apoe_col else None),
        "status": status,
    }

per_study = [check_apoe(s, m) for s, m in STUDY_METADATA.items()]

checked = [r for r in per_study if r["status"] != "skipped"]
if not checked:
    overall = "incomplete"
elif all(r["status"] == "pass" for r in checked):
    overall = "pass"
elif any(r["status"] == "fail" for r in checked):
    overall = "fail"
else:
    overall = "warn"

audit_b = {"audit": "B — donor APOE metadata", "status": overall, "per_study": per_study}

with open(AUDIT_REPORT_PATH) as f:
    report = json.load(f)
report["audit_b_apoe_metadata"] = audit_b
with open(AUDIT_REPORT_PATH, "w") as f:
    json.dump(report, f, indent=2)

print(json.dumps(audit_b, indent=2))

## 6 — Handoff to colab_01

Prints a final summary and the explicit `git add / commit / push` commands needed to land the env snapshot + audit report on GitHub before the runtime resets. `colab_01_li2025_qc.ipynb` is the next notebook — vertical slice on the smallest study (Li 2025, 343k nuclei, GEO GSE237718) before scaling to SEA-AD + Haney 2024.

### 6a — Summary + commit instructions

Reads back `audit_report.json` and lists artifact paths, then **prints** the `git add / commit / push` commands to run. Push is left manual on purpose: git auth in Colab is the fragile step, and running it yourself surfaces git's own error directly instead of burying it in a cell traceback. Copy the three printed lines into a new cell (with a `!` prefix each) and run them to land the artifacts on GitHub before the runtime resets.

In [ ]:
import json, os, shlex, subprocess

with open(AUDIT_REPORT_PATH) as f:
    report = json.load(f)

print("=== Audit summary ===")
for k, v in report.items():
    print(f"  {k}: {v.get('status')}")

print("\n=== Artifacts written ===")
for p in [FREEZE_PATH, ENV_JSON_PATH, AUDIT_REPORT_PATH]:
    print(f"  {p}")

print("\n=== Commit + push (run from a fresh cell if these fail) ===")
rel_paths = [os.path.relpath(p, REPO_PATH) for p in [FREEZE_PATH, ENV_JSON_PATH, AUDIT_REPORT_PATH]]
msg = f"colab_00: env capture + preemptive audits F, B ({TODAY})"
cmds = [
    f"cd {REPO_PATH} && git add {' '.join(rel_paths)}",
    f"cd {REPO_PATH} && git commit -m {__import__('shlex').quote(msg)}",
    f"cd {REPO_PATH} && git push",
]
for c in cmds:
    print(f"  {c}")

print("\nNext: colab_01_li2025_qc.ipynb (Li 2025 download + secondary_qc + Audit A on this study).")